# CS781 Phase 3 — A4: LORMA (Updated)

**One notebook is enough.** Open this `.ipynb` in Kaggle and run cells from top to bottom. No ZIP or other Python source files are necessary.

The notebook downloads my original Phase 1 weights and matching tokenizer from Hugging Face, then downloads the gated official CL/IT Phase 3 data with my **HF** Kaggle secret.

**A4:** Keep the original mixed local/dilated causal attention, and train LoRMA+ adapters on its attention projections.

Every experiment saves the best development Macro-F1 checkpoint, detailed results, `prediction.json`, and `submission.zip`, and the last cell uploads results privately to Hugging Face.

**Important:** The code has offline/synthetic checks, not a verified live GPU score; run on Kaggle to measure real results.

## 1. Install packages
Kaggle already has PyTorch. We only install the Hugging Face and evaluation packages used here. **No PEFT or torchao required** (avoids the Phase 2 dependency issue).


In [ ]:
%pip -q install "transformers>=4.46,<5" "huggingface_hub>=0.31,<2" "safetensors>=0.4.5" "scikit-learn>=1.4" "tqdm>=4.66"


## 2. Connect safely to my Hugging Face account
I load my secret from Kaggle; this notebook never prints it. The login also lets me download private checkpoints and later upload the trained Phase 3 result.


In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, get_token, login

# Kaggle secret should be named HF. In Colab, use notebook_login() first.
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF') or HF_TOKEN
except Exception:
    pass
HF_TOKEN = HF_TOKEN or get_token()
if not HF_TOKEN:
    raise RuntimeError('No Hugging Face token. Add Kaggle Secret HF (enable notebook access), then restart.')

# Sign in for this Kaggle Python session. Browser login alone is not sufficient.
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
MY_HF_NAME = api.whoami()['name']
print('Logged into Hugging Face account:', MY_HF_NAME)


## 3. My training settings

The settings below are all in one place so I can understand and change them.

The main improvements compared with my first notebook are **up to 6 epochs**, **early stopping**, **192-token inputs**, and modest **rare-label balancing**.

A3: teacher and student have separate learning rates; the student changes more pretrained parameters, so it uses a smaller rate.

A4: keep LoRMA but adapt more of the attention layer (Q/K/V/output).

If Kaggle runs out of memory, first lower `BATCH_SIZE` to `1`, then `MAX_LENGTH` to `128`.

In [ ]:
# PHASE3_SOURCE_CELL
from pathlib import Path
import torch

ARCH = 'a4'
EXPERIMENT = 'a4_lorma_improved'
PHASE1_REPO = 'P-261110044/cs781-a4-checkpoints'
PHASE1_WEIGHTS = 'final_model/model_state.pt'
OUTPUT_REPO = MY_HF_NAME + '/cs781-phase3-a4-lorma-improved'

# Use Kaggle's persistent output folder; otherwise use the local working folder.
OUTPUT_DIR = Path('/kaggle/working/phase3_a4_improved') if Path('/kaggle/working').exists() else Path.cwd() / 'phase3_a4_improved'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Start small. If out-of-memory, lower BATCH_SIZE to 1 and MAX_LENGTH to 96.
EPOCHS = 6                # Maximum epochs; early stopping can stop sooner
BATCH_SIZE = 2           # Reduce to 1 if the GPU runs out of memory
GRAD_ACCUM = 4           # Accumulate gradients to emulate a larger batch
MAX_LENGTH = 192         # Total tokens for context and the target sentence
LEARNING_RATE = 2e-4    # Main adapter learning rate

config = {
    'out_dir': str(OUTPUT_DIR),
    'hf_token': HF_TOKEN,
    'phase1_sources': {ARCH: {
        'repo': PHASE1_REPO,
        'weight': PHASE1_WEIGHTS,
        'tokenizer_dirs': ['final_model/', 'architecture-4-final/', ''],
        'config_files': ['final_model/architecture_config.json', 'architecture_config.json'],
        'local': '',  # empty means use Hugging Face; no manual files needed
    }},
    'dataset_repo': 'Exploration-Lab/CS-781-Capstone',
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'epochs': EPOCHS,
    'max_length': MAX_LENGTH,
    'lr': LEARNING_RATE,
    'use_context': True,  # previous text + next text BEFORE current target sentence
    'patience': 2,
    'use_balanced_batches': True,  # mild rare-label resampling during training only
    'seed': 42,
}
print('Experiment:', EXPERIMENT)
print('My trained Phase 1 source:', PHASE1_REPO, '/', PHASE1_WEIGHTS)
print('GPU available:', torch.cuda.is_available())


## 4. My original Phase 1 A4 architecture
This is the actual model definition from the Phase 1 notebook—not a new pretrained model downloaded from a library. Its trained weights will be loaded later from Hugging Face. **Keep this cell unchanged** so the saved checkpoint matches.


In [ ]:
# PHASE3_SOURCE_CELL
# Extracted from your original Architecture4.ipynb model-definition cell.
# Original architecture code preserved, not replaced with library model.

import torch

import torch.nn as nn

import torch.nn.functional as F

from transformers.modeling_outputs import CausalLMOutput

def build_pattern_mask(context_length, pattern_type, pattern_value=None):
    """
    Build a boolean causal attention mask of shape [context_length, context_length].

    PyTorch SDPA interprets True as "this query may attend to this key".

    Example for dilation=2 at query position t:
        allowed keys = t, t-2, t-4, t-6, ...
    """
    query_pos = torch.arange(context_length).unsqueeze(1)  # [T, 1]
    key_pos = torch.arange(context_length).unsqueeze(0)    # [1, T]

    distance = query_pos - key_pos
    causal = distance >= 0

    if pattern_type == "global":
        allowed = causal

    elif pattern_type == "local":
        window = int(pattern_value)
        assert window >= 1
        allowed = causal & (distance < window)

    elif pattern_type == "dilated":
        dilation = int(pattern_value)
        assert dilation >= 1
        allowed = causal & ((distance % dilation) == 0)

    else:
        raise ValueError(f"Unknown attention pattern: {pattern_type}")

    return allowed

class PatternCausalSelfAttention(nn.Module):
    """Causal self-attention with a chosen local/dilated/global pattern."""

    def __init__(
        self,
        dim,
        num_heads,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.context_length = context_length
        self.pattern_type = pattern_type
        self.pattern_value = pattern_value
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        # The mask is fixed by the architecture, so build it only once.
        # persistent=False means this large mask is NOT written into checkpoints.
        pattern_mask = build_pattern_mask(
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
        )
        self.register_buffer(
            "pattern_mask",
            pattern_mask,
            persistent=False,
        )

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        if past_kv is None:
            # Training / prompt prefill.
            k_all = k
            v_all = v
            key_length = T
            query_start = 0

        else:
            # Incremental generation: append the new token's K/V to the cache.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv

            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            key_length = k_all.size(2)
            query_start = key_length - T

        if key_length > self.context_length:
            raise ValueError(
                f"Attention cache length {key_length} exceeds "
                f"context length {self.context_length}."
            )

        # Slice exactly the query rows and key columns needed in this call.
        # Shape becomes [1, 1, query_length, key_length] and broadcasts over
        # batch and attention heads.
        attn_mask = self.pattern_mask[
            query_start:query_start + T,
            :key_length,
        ][None, None, :, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k_all,
            v_all,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,  # causality is already encoded in attn_mask
        )

        new_kv = (k_all, v_all) if use_cache else None

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, self.dim)
        )

        return self.o_proj(attn_out), new_kv

class MultiScaleTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        ffn_dim,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()

        # Pre-LayerNorm Transformer block.
        self.norm1 = nn.LayerNorm(dim)
        self.attn = PatternCausalSelfAttention(
            dim=dim,
            num_heads=num_heads,
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(
            self.norm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )

        # Residual connection around attention.
        x = x + self.dropout(attn_out)

        # Residual connection around feed-forward network.
        x = x + self.dropout(self.ffn(self.norm2(x)))

        return x, new_kv

class MultiScaleDilatedLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length=2048,
        dim=600,
        num_heads=10,
        ffn_dim=1600,
        attention_patterns=None,
        dropout=0.1,
    ):
        super().__init__()

        if attention_patterns is None:
            raise ValueError("attention_patterns must be provided.")

        self.vocab_size = vocab_size
        self.context_length = context_length
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.attention_patterns = attention_patterns

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(context_length, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            MultiScaleTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                context_length=context_length,
                pattern_type=pattern["type"],
                pattern_value=pattern.get("value"),
                dropout=dropout,
            )
            for pattern in attention_patterns
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Initialize first, then tie token-embedding and output weights.
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T = input_ids.shape

        if position_offset + T > self.context_length:
            raise ValueError(
                f"Sequence position {position_offset + T} exceeds "
                f"context length {self.context_length}."
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device,
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        x = self.embed_dropout(x)

        if past_key_values is None:
            layer_past = [None] * len(self.blocks)
        else:
            if len(past_key_values) != len(self.blocks):
                raise ValueError("Wrong number of layer caches.")
            layer_past = past_key_values

        new_caches = []

        for block, past_kv in zip(self.blocks, layer_past):
            x, new_kv = block(
                x,
                past_kv=past_kv,
                use_cache=use_cache,
            )

            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)

        return x, (new_caches if use_cache else None)

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        # Training examples contain full fixed-length token blocks, so no
        # padding mask is needed here. The architectural causal/sparse mask is
        # handled inside each attention layer.
        hidden, _ = self._run_hidden(input_ids)
        logits = self.lm_head(hidden)

        loss = None

        if labels is not None:
            # Next-token language modelling:
            # position t predicts label at position t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits,
        )

    @torch.inference_mode()
    def prefill(self, input_ids):
        """Process the full prompt once and create K/V caches for all layers."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )

        return self.lm_head(hidden), caches

    @torch.inference_mode()
    def decode_step(
        self,
        input_ids,
        past_key_values,
        position_offset,
    ):
        """Process one newly generated token using the existing K/V caches."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )

        logits = self.lm_head(hidden[:, -1, :])
        return logits, caches


## 5. Phase 3 common helper functions (included inside this notebook)
The next sections contain the small support functions previously stored in `phase3_core.py`. They are now **inside this notebook**. Read the comments if you want to understand what each step does; otherwise run them in order.


### 5.1 Helper imports and official file names
These imports and constants define the official Phase 3 filenames. No outside legal data is used.


In [ ]:
# PHASE3_SOURCE_CELL
import copy
import gc
import json
import math
import os
import random
import re
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download
from transformers import PreTrainedTokenizerFast


OFFICIAL_DATASET = 'Exploration-Lab/CS-781-Capstone'
PHASE3_DIR = 'Phase_3_RR'
TRAIN_FILES = ['Train/CL_train.json', 'Train/IT_train.json']
DEV_FILES = ['Train/CL_dev.json', 'Train/IT_dev.json']
TEST_FILES = ['Test/CL_test.json', 'Test/IT_test.json']
LABELS_FILE = 'Train/label_vocab.json'
# Exactly four required submissions, covering A1/A2/A3/A4.
# A3 also trains a teacher, but only the distilled student counts as its submission.
ARCH_BY_EXPERIMENT = {
    'a1_lora': 'a1',
    'a2_ia3': 'a2',
    'a3_teacher': 'a3',  # helper checkpoint for the distillation run
    'a3_kd': 'a3',
    'a3_kd_improved': 'a3',
    'a4_lorma': 'a4',
    'a4_lorma_improved': 'a4',
}
ALL_EXPERIMENTS = ['a1_lora', 'a2_ia3', 'a3_kd', 'a4_lorma']
A4_PATTERNS = [
    {'type':'local','value':256}, {'type':'dilated','value':2},
    {'type':'local','value':512}, {'type':'dilated','value':4},
    {'type':'local','value':512}, {'type':'dilated','value':8},
    {'type':'local','value':1024}, {'type':'global','value':None},
]


### 5.2 Download documents and read the official labels
We fetch CL/IT training, development and test documents and check the ID and label formats. We never use private test labels.


In [ ]:
# PHASE3_SOURCE_CELL
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def read_json(path):
    with open(path, 'r', encoding='utf8') as f: return json.load(f)


def download_data(root, repo_id=OFFICIAL_DATASET):
    """Download ONLY Phase 3 data; never download private reference labels."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    result = {}
    for rel in TRAIN_FILES + DEV_FILES + TEST_FILES + [LABELS_FILE]:
        filename = PHASE3_DIR + '/' + rel
        try:
            p = hf_hub_download(repo_id=repo_id, filename=filename, repo_type='dataset', token=HF_TOKEN)
        except Exception as e:
            raise RuntimeError(f'Cannot download {filename} from {repo_id}. '
                               f'Check that the HF Kaggle secret belongs to an account approved for the gated dataset. 'f'Original error: {e}') from e
        result[rel] = Path(p)
    return result


def extract_labels(raw):
    """Accept documented label_vocab.json with a list or label-id dictionary."""
    if isinstance(raw, list):
        if all(isinstance(s,str) for s in raw): labels=raw
        elif all(isinstance(x,dict) for x in raw): labels=[str(x.get('label',x.get('name'))) for x in raw]
        else: raise ValueError('Unknown list label vocabulary structure')
    elif isinstance(raw,dict):
        if 'labels' in raw: return extract_labels(raw['labels'])
        if 'label_vocab' in raw: return extract_labels(raw['label_vocab'])
        if 'label2id' in raw: return [k for k,_ in sorted(raw['label2id'].items(),key=lambda x:int(x[1]))]
        if 'id2label' in raw: return [v for k,v in sorted(raw['id2label'].items(),key=lambda x:int(x[0]))]
        if all(str(k).isdigit() for k in raw): labels=[v for k,v in sorted(raw.items(),key=lambda x:int(x[0]))]
        elif all(isinstance(v,int) for v in raw.values()): labels=[k for k,v in sorted(raw.items(),key=lambda x:x[1])]
        else: raise ValueError(f'Unknown vocabulary shape: {list(raw)[:8]}')
    else: raise TypeError('Unknown vocabulary type')
    assert labels and len(labels)==len(set(labels)) and all(isinstance(x,str) for x in labels)
    return labels


def normalize_documents(raw, split_name, must_have_labels):
    """Retain official string IDs and original sentence order."""
    if isinstance(raw,dict):
        if 'id' in raw and 'text' in raw:
            # A single standalone document, for small tests or unusual exports.
            raw=[raw]
        else:
            for key in ('documents','data','examples'):
                if key in raw and isinstance(raw[key],(list,dict)):
                    raw=raw[key]
                    break
            # Official data can be either a list or an ID -> document mapping.
            if isinstance(raw,dict):
                raw=[{'id':key,**(value if isinstance(value,dict) else {'text':value})}
                     for key,value in raw.items()]
    if not isinstance(raw,list): raise TypeError(f'{split_name}: expected list/dict')
    docs=[]; seen=set()
    for i, doc in enumerate(raw):
        if not isinstance(doc,dict): raise TypeError(f'{split_name} document {i} is not a dict')
        did=str(doc.get('id',doc.get('doc_id',doc.get('document_id',''))))
        if not did: raise ValueError(f'{split_name}: missing official document id at item {i}')
        if did in seen: raise ValueError(f'{split_name}: duplicate document id {did}')
        seen.add(did)
        sentences=doc.get('text',doc.get('sentences'))
        if not isinstance(sentences,list) or not all(isinstance(t,str) for t in sentences):
            raise TypeError(f'{split_name}: {did} must have a list of sentence strings in text')
        item={'id':did,'text':sentences}
        labels=doc.get('labels',doc.get('label'))
        if must_have_labels:
            if not isinstance(labels,list) or len(labels)!=len(sentences):
                raise ValueError(f'{split_name}: {did} sentence/label lengths differ')
            item['labels'] = [str(v) for v in labels]
        docs.append(item)
    return docs


def load_splits(paths):
    classes=extract_labels(read_json(paths[LABELS_FILE]))
    idx={v:i for i,v in enumerate(classes)}
    splits={}
    for group,files,labeled in [('train',TRAIN_FILES,True),('dev',DEV_FILES,True),('test',TEST_FILES,False)]:
        docs=[]
        for rel in files:
            domain=rel.split('/')[-1].split('_')[0]
            part=normalize_documents(read_json(paths[rel]),rel,labeled)
            for d in part:
                d['domain']=domain
                if labeled:
                    for label in d['labels']:
                        if label not in idx: raise ValueError(f'Unknown label {label} in {rel}')
            docs.extend(part)
        if len({d['id'] for d in docs})!=len(docs):
            raise ValueError(f'Duplicate ids across CL/IT {group}. Do not silently overwrite.')
        splits[group]=docs
    return splits,classes


def make_examples(docs,labeled=True):
    return [{'doc_id':doc['id'], 'domain':doc['domain'],
             'sentences':doc['text'], 'sent_index':i,
             'sentence':sentence, 'label':doc['labels'][i] if labeled else None}
            for doc in docs for i,sentence in enumerate(doc['text'])]


### 5.3 Preparing sentence input (causal Transformer)

We label **one sentence at a time**. For a causal model, earlier tokens can influence later tokens, so we place the surrounding context *before* the current sentence. The classifier uses the last token of the current sentence. We never provide neighboring gold labels.

Format: `[previous text] </s> [next text] </s> [CURRENT SENTENCE]`. If there is insufficient room, we keep as much of the current sentence as possible.

In [ ]:
# PHASE3_SOURCE_CELL
def encode_example(ex, tok, max_len=192, use_context=True):
    """Convert one sentence and its optional neighbors into token IDs.

    IMPORTANT: A3/A4 are causal (left-to-right). To allow the target sentence
    to use the NEXT sentence as extra context, we place next-sentence tokens
    BEFORE target-sentence tokens. No gold labels are included.
    """
    current = tok.encode(ex['sentence'], add_special_tokens=False)
    if not current:
        current = [tok.unk_token_id if tok.unk_token_id is not None else 0]
    # Keep the target sentence first in priority: don't discard all its text.
    current = current[:max_len]
    if not use_context or len(current) >= max_len - 2:
        return current, 0, len(current)
    sep = tok.eos_token_id
    if sep is None:
        raise ValueError('The original Phase 1 tokenizer has no EOS token')

    index = ex['sent_index']
    text = ex['sentences']
    previous = tok.encode(text[index-1], add_special_tokens=False) if index > 0 else []
    following = tok.encode(text[index+1], add_special_tokens=False) if index+1 < len(text) else []

    # Two separators at most. Divide remaining space between neighbors.
    available = max_len - len(current) - int(bool(previous)) - int(bool(following))
    before = min(len(previous), max(0, available // 2))
    after = min(len(following), max(0, available - before))
    before = min(len(previous), max(0, available - after))

    previous = previous[-before:] if before else []
    following = following[:after] if after else []
    prefix = (previous + [sep] if previous else []) + (following + [sep] if following else [])
    result = prefix + current
    assert len(result) <= max_len
    return result, len(prefix), len(result)

class RRClassifier(nn.Module):
    """Add a small classification head atop a Phase-1 model, NOT a replacement model."""
    def __init__(self,backbone,arch,dim,nlabels):
        super().__init__()
        self.backbone=backbone
        self.arch=arch
        self.classifier=nn.Sequential(nn.Dropout(0.1),nn.Linear(dim,nlabels))

    def hidden_for_one(self, ids, current_start):
        # ids: shape [1,T]. Trim padding altogether, important for bidirectional a2.
        b=self.backbone
        if self.arch=='a1':
            seq=ids.size(1)
            h=b.token_embedding(ids)+b.position_embedding(torch.arange(seq,device=ids.device))
            causal=torch.triu(torch.ones(seq,seq,dtype=torch.bool,device=ids.device),diagonal=1)
            for block in b.blocks: h=block(h,causal)
            h=b.final_norm(h)
        elif self.arch=='a2':
            h=b.encode(ids)  # bidirectional encoder, no autoregressive decoder needed
        elif self.arch in ('a3','a4'):
            h,_=b._run_hidden(ids)
        else: raise ValueError(self.arch)
        # Decoder models use last sentence token, encoder model pools entire sentence.
        if self.arch=='a2': return h[:,current_start:,:].mean(dim=1)
        return h[:,-1,:]

    def forward(self, batch_ids, starts):
        if self.arch == 'a2':
            # Bidirectional encoder: process without padding so padded tokens
            # never affect the attention results for real sentence tokens.
            states = [self.hidden_for_one(ids[None, :], start)
                      for ids, start in zip(batch_ids, starts)]
            return self.classifier(torch.cat(states, dim=0))

        # Causal architectures: right padding cannot affect earlier tokens.
        # Process a whole batch in one Transformer pass to save GPU time.
        lengths = [len(ids) for ids in batch_ids]
        seq = torch.zeros(len(batch_ids), max(lengths),
                          dtype=torch.long, device=batch_ids[0].device)
        for i, ids in enumerate(batch_ids):
            seq[i, :len(ids)] = ids
        b = self.backbone
        if self.arch == 'a1':
            t = seq.size(1)
            h = b.token_embedding(seq) + b.position_embedding(
                torch.arange(t, device=seq.device))
            mask = torch.triu(torch.ones(t, t, dtype=torch.bool,
                                         device=seq.device), diagonal=1)
            for block in b.blocks:
                h = block(h, mask)
            h = b.final_norm(h)
        elif self.arch in ('a3', 'a4'):
            h, _ = b._run_hidden(seq)
        else:
            raise ValueError(self.arch)
        last_positions = torch.tensor(lengths, device=seq.device)-1
        states = h[torch.arange(len(lengths), device=seq.device), last_positions]
        return self.classifier(states)


### 5.4 LORMA+ adapters on more attention projections

The original A4 attention patterns and pretrained Q/K/V/output weights are unchanged. I add small trainable multiplicative low-rank changes to **Q, K, V and output**. The pretrained base weights remain frozen.

In [ ]:
# PHASE3_SOURCE_CELL
class LoRALinear(nn.Module):
    """LoRA: W = W0 + alpha/r * B @ A."""
    def __init__(self,base,rank=8,alpha=8):
        super().__init__(); self.base=base
        for p in self.base.parameters():p.requires_grad=False
        self.A=nn.Linear(base.in_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        self.scale=alpha/rank
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
    def forward(self,x): return self.base(x)+self.B(self.A(x))*self.scale


class LoRMAPlusLinear(nn.Module):
    """LoRMA+ (Bihany et al., ACL 2025): (I + alpha/r * BA) W0 x.
    Apply to zero-bias q/o projections only: the formula is exact.
    """
    def __init__(self,base,rank=8,alpha=8):
        super().__init__(); assert base.bias is None, 'LoRMA targets bias-free projections'
        self.base=base
        for p in self.base.parameters():p.requires_grad=False
        self.A=nn.Linear(base.out_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        self.scale=alpha/rank
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5));nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x)
        return y+self.scale*self.B(self.A(y))


class IA3FeedForward(nn.Module):
    """IA3 FFN intermediate-channel activation scaling (frozen Linear + learned vector)."""
    def __init__(self,base):
        super().__init__();self.base=base
        for p in self.base.parameters():p.requires_grad=False
        self.scales=nn.Parameter(torch.ones(base.out_features))
    def forward(self,x):return self.base(x)*self.scales


def replace_modules(model,matcher,wrapper):
    found=[]
    # Snapshot names before replacing modules to avoid recursive traversal into adapters.
    for name,mod in list(model.named_modules()):
        if name and isinstance(mod,nn.Linear) and matcher(name,mod):
            owner=model
            parts=name.split('.')
            for p in parts[:-1]: owner=owner[int(p)] if p.isdigit() else getattr(owner,p)
            last=parts[-1]
            if last.isdigit(): owner[int(last)]=wrapper(mod)
            else: setattr(owner,last,wrapper(mod))
            found.append(name)
    if not found: raise RuntimeError('No module matched adapter target; check architecture definition')
    return found


def make_adaptation(model, experiment):
    """Four genuinely different methods, with trainable head in every experiment.

    A1: LoRA on feed-forward linear layers (A1 MultiheadAttention has fused QKV,
        so blindly targeting q_proj/v_proj, as for A3, would silently do nothing).
    A2: IA3 activation-channel scales in encoder feed-forward layers.
    A3: trained LoRA teacher and smaller supervised + soft-label KD student.
    A4: multiplicative LoRMA applied to attention projections (same as Phase 2).
    """
    for p in model.backbone.parameters():
        p.requires_grad = False
    if experiment == 'a1_lora':
        return replace_modules(model.backbone,
            lambda name, layer: '.feed_forward.' in name and
            name.rsplit('.', 1)[-1] in ('0', '2'),
            lambda layer: LoRALinear(layer, rank=8, alpha=16))
    if experiment == 'a2_ia3':
        return replace_modules(model.backbone,
            lambda name, layer: name.startswith('encoder_blocks.') and
            name.endswith('feed_forward.0'),
            lambda layer: IA3FeedForward(layer))
    if experiment == 'a3_teacher':
        return replace_modules(model.backbone,
            lambda name, layer: name.endswith(('.q_proj', '.v_proj')) and layer.bias is None,
            lambda layer: LoRALinear(layer, rank=8, alpha=16))
    if experiment == 'a3_kd':
        for param in model.parameters():
            param.requires_grad = True
        return ['student: first four pretrained A3 blocks + new RR head']
    if experiment in ('a4_lorma', 'a4_lorma_improved'):
        # Keep LoRMA+, but adapt Q, K, V AND output projection.
        # Each projection is bias-free in my original A4 architecture.
        return replace_modules(model.backbone,
            lambda name, layer: name.endswith(('.q_proj', '.k_proj', '.v_proj', '.o_proj')) and layer.bias is None,
            lambda layer: LoRMAPlusLinear(layer, rank=8, alpha=16))
    raise ValueError('Unknown experiment: ' + experiment)


### 5.5 Hugging Face checkpoint loading and exact model reconstruction
We download the exact Phase 1 checkpoint and its original tokenizer, infer sizes from saved tensors and load weights strictly. This prevents accidental random initialization.


In [ ]:
# PHASE3_SOURCE_CELL
def load_phase1_files(arch, config):
    """Download *exactly* the Phase 1 checkpoint identified in your Phase 2 notebook.

    Phase 2 lessons:
      * Do not use random pretrained-model defaults instead of actual saved weights.
      * Do not silently fall back to an older checkpoint.
      * Never pair weights with a different BPE tokenizer.
      * Keep the architecture config containing attention heads/window patterns.
    """
    source = config['phase1_sources'][arch]
    repo = source['repo']
    cache = Path(config['out_dir']) / 'phase1_download' / arch
    cache.mkdir(parents=True, exist_ok=True)
    local = source.get('local', '')
    token = config.get('hf_token')
    api = HfApi(token=token)
    if local:
        root = Path(local)
        if not root.exists():
            raise FileNotFoundError('Local checkpoint folder not found: ' + str(root))
        names = [str(q.relative_to(root)) for q in root.rglob('*') if q.is_file()]
    else:
        names = api.list_repo_files(repo_id=repo, repo_type='model', token=token)
    # Selecting the exact checkpoint avoids unwanted checkpoint fallback.
    weight_rel = source['weight']
    if weight_rel not in names:
        raise FileNotFoundError(f'Exact Phase 1 checkpoint {weight_rel} not found in {repo}. '
                                f'First few remote files: {names[:10]}')

    def fetch(remote_name):
        if local:
            return root / remote_name
        return Path(hf_hub_download(repo_id=repo, filename=remote_name,
                      repo_type='model', token=token, local_dir=str(cache)))

    weight = fetch(weight_rel)
    # Look for the BPE tokenizer used with this exact model.
    tokenizer_dir = None
    for folder in source['tokenizer_dirs']:
        full = folder.rstrip('/') + ('/' if folder else '')
        if full + 'tokenizer.json' in names:
            tokenizer_dir = full
            break
    if tokenizer_dir is None:
        raise FileNotFoundError(f'No tokenizer.json in expected paths of {repo}. '
                                'Do not use a different tokenizer.')
    for filename in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json'):
        remote = tokenizer_dir + filename
        if remote in names: fetch(remote)
    tok_root = (root if local else cache) / tokenizer_dir
    tok_file = tok_root / 'tokenizer.json'
    if not tok_file.is_file():
        raise FileNotFoundError(f'Tokenizer file missing: {tok_file}')

    # Use the same BPE loader and special tokens as your Phase 1 notebooks.
    # Loading the JSON directly avoids the TokenizersBackend class warning.
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=str(tok_file),
        pad_token='<pad>',
        unk_token='<unk>',
        bos_token='<s>',
        eos_token='</s>'
    )
    if not tokenizer.is_fast:
        raise ValueError('Expected the original fast BPE tokenizer')

    # Check that the special tokens exist in the ORIGINAL JSON file.
    # If a token is missing, the loader could add a new token ID.
    # That would make the Phase 1 embedding weights incompatible.
    saved_tok = read_json(tok_file)
    original_tokens = set(saved_tok.get('model', {}).get('vocab', {}))
    original_tokens.update(x['content'] for x in saved_tok.get('added_tokens', []))
    for special in ('<pad>', '<unk>', '<s>', '</s>'):
        if special not in original_tokens:
            raise ValueError(f'Missing {special} in original tokenizer.json')

    print('Tokenizer class:', type(tokenizer).__name__)
    print('Tokenizer vocabulary size:', len(tokenizer))
    print('Special token IDs:', tokenizer.all_special_ids)
    cfg = {}
    for filename in source['config_files']:
        if filename in names:
            cfg = read_json(fetch(filename))
            break
    return weight, tokenizer, cfg


def _read_state(path):
    if str(path).endswith('.safetensors'):
        from safetensors.torch import load_file
        state = load_file(str(path), device='cpu')
    else:
        state = torch.load(str(path), map_location='cpu', weights_only=True)
    # The Phase 1 notebooks saved model_state.pt, others Trainer model.safetensors.
    if isinstance(state, dict):
        for key in ('model_state', 'state_dict', 'model_state_dict', 'model'):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break
    if not isinstance(state, dict): raise ValueError('Invalid Phase 1 state dictionary')
    for prefix in ('module.', '_orig_mod.'):
        if state and all(k.startswith(prefix) for k in state):
            state = {k[len(prefix):]:v for k,v in state.items()}
    if 'token_embedding.weight' not in state:
        raise ValueError('Missing Phase 1 token_embedding.weight')
    return state


def _blocks(state, name):
    numbers = {int(m.group(1)) for k in state
                if (m := re.match(re.escape(name) + r'\.(\d+)\.', k))}
    if not numbers or numbers != set(range(max(numbers)+1)):
        raise ValueError('Missing/nonconsecutive Phase 1 blocks: ' + name)
    return len(numbers)


def actual_model_params(arch, state, cfg, tokenizer_size):
    """Architecture sizes from checkpoint tensors, not notebook defaults."""
    vocab, dim = state['token_embedding.weight'].shape
    if vocab != tokenizer_size:
        raise ValueError(f'Wrong BPE tokenizer: {tokenizer_size}, checkpoint requires {vocab}')
    if 'vocab_size' in cfg and int(cfg['vocab_size']) != vocab:
        raise ValueError('Saved architecture config disagrees with checkpoint vocabulary size')
    for field in ('d_model', 'dim'):
        if field in cfg and int(cfg[field]) != dim:
            raise ValueError('Saved architecture config disagrees with checkpoint hidden dimension')
    if arch == 'a1':
        layers = _blocks(state,'blocks')
        heads = int(cfg.get('n_heads', cfg.get('num_heads', 10)))
        if (dim,layers) != (640,8) and not ('n_heads' in cfg or 'num_heads' in cfg):
            raise ValueError('A1 head count unknown for non-original dimensions')
        args=dict(vocab_size=vocab,context_length=state['position_embedding.weight'].shape[0],
                  d_model=dim,n_heads=heads,n_layers=layers,
                  d_ff=state['blocks.0.feed_forward.0.weight'].shape[0],dropout=.1)
    elif arch == 'a2':
        n_encoder = _blocks(state,'encoder_blocks')
        n_decoder = _blocks(state,'decoder_blocks')
        heads = int(cfg.get('n_heads', cfg.get('num_heads', 15)))
        if (dim,n_encoder)!=(600,8) and not ('n_heads' in cfg or 'num_heads' in cfg):
            raise ValueError('A2 head count unknown for non-original dimensions')
        args=dict(vocab_size=vocab,source_length=state['source_position.weight'].shape[0],
                  target_length=state['target_position.weight'].shape[0],d_model=dim,
                  n_heads=heads,n_encoder_layers=n_encoder,n_decoder_layers=n_decoder,
                  d_ff=state['encoder_blocks.0.feed_forward.0.weight'].shape[0],dropout=.1)
    elif arch == 'a3':
        if 'num_layers' in cfg and int(cfg['num_layers']) != _blocks(state,'blocks'):
            raise ValueError('A3 saved config layer count disagrees with checkpoint')
        needed=['num_heads','num_kv_heads','window_size','chunk_size']
        if any(k not in cfg for k in needed):
            raise ValueError('A3 missing architecture_config.json settings: '+str(needed))
        args=dict(vocab_size=vocab,dim=dim,num_layers=_blocks(state,'blocks'),
                  num_heads=int(cfg['num_heads']),num_kv_heads=int(cfg['num_kv_heads']),
                  ffn_dim=state['blocks.0.ffn.gate_proj.weight'].shape[0],
                  window_size=int(cfg['window_size']),chunk_size=int(cfg['chunk_size']),
                  dropout=float(cfg.get('dropout',.1)))
    elif arch == 'a4':
        if 'attention_patterns' not in cfg or 'num_heads' not in cfg:
            raise ValueError('A4 missing saved attention patterns/num_heads')
        if len(cfg['attention_patterns'])!=_blocks(state,'blocks'):
            raise ValueError('A4 attention pattern count differs from checkpoint layers')
        args=dict(vocab_size=vocab,context_length=state['position_embedding.weight'].shape[0],
                  dim=dim,num_heads=int(cfg['num_heads']),
                  ffn_dim=state['blocks.0.ffn.0.weight'].shape[0],
                  attention_patterns=cfg['attention_patterns'],dropout=float(cfg.get('dropout',.1)))
    else: raise ValueError(arch)
    if dim % args.get('n_heads', args.get('num_heads',1)):
        raise ValueError('Invalid checkpoint attention head count')
    return args, dim


class Phase1A2Encoder(nn.Module):
    """Load *only* the pretrained A2 encoder, exactly like your Phase 2 A2 notebook.

    The A2 decoder and 32k-word vocabulary output projection are never used
    for sentence classification. Dropping them makes Kaggle training cheaper.
    The encoder names are identical to the original Phase 1 checkpoint names.
    """
    only_encoder = True

    def __init__(self, vocab_size, source_length, d_model, n_heads,
                 n_encoder_layers, d_ff, dropout=0.1, **unused_decoder_settings):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.source_position = nn.Embedding(source_length, d_model)
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_encoder_layers)
        ])

    def encode(self, input_ids):
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.token_embedding(input_ids) + self.source_position(positions)
        for layer in self.encoder_blocks:
            x = layer(x)
        return x

def instantiate(arch, args, student=False):
    """Load original model class; student starts from its first pretrained layers."""
    args = dict(args)
    if student:
        if arch!='a3': raise ValueError('Only A3 is used for distillation')
        args['num_layers']=max(1, min(4,args['num_layers']-1))
    if arch=='a1': return DecoderTransformerLM(**args)
    if arch=='a2': return Phase1A2Encoder(**args)
    if arch=='a3': return ModernLocalTransformerLM(**args)
    if arch=='a4': return MultiScaleDilatedLM(**args)
    raise ValueError(arch)


def load_pretrained(backbone,state,student=False):
    """Strict loading: only known tied LM-head aliases are normalized."""
    state=dict(state)
    expected=backbone.state_dict()
    if hasattr(backbone, 'lm_head'):
        for key,other in [('lm_head.weight','token_embedding.weight'),
                          ('token_embedding.weight','lm_head.weight')]:
            if key in expected and key not in state and other in state:
                state[key]=state[other]
    elif 'lm_head.weight' in state and 'lm_head.weight' not in expected:
        if 'output_layer.weight' not in state:
            state['output_layer.weight']=state['lm_head.weight']
        del state['lm_head.weight']
    if student or getattr(backbone,'only_encoder',False):
        # A2: take encoder weights only (Phase 2 A2 strategy).
        # A3 student: exclude upper teacher layers only after aliases are fixed.
        state={key:state[key] for key in expected if key in state}
    missing = set(expected) - set(state)
    extra = set(state) - set(expected)
    wrong=[k for k in set(expected)&set(state) if expected[k].shape!=state[k].shape]
    if missing or extra or wrong:
        raise ValueError(f'Phase 1 weights do not match architecture: '
                         f'missing={list(missing)[:4]}, extra={list(extra)[:4]}, wrong shape={wrong[:4]}')
    backbone.load_state_dict(state,strict=True)
    return True


def prepare_classifier(arch, experiment, config, student=False):
    weight, tokenizer, saved_cfg = load_phase1_files(arch,config)
    state = _read_state(weight)
    args, dim = actual_model_params(arch,state,saved_cfg,len(tokenizer))
    # save provenance so you can defend which checkpoint really trained the classifier
    work = Path(config['out_dir']) / experiment
    work.mkdir(parents=True,exist_ok=True)
    metadata={'architecture':arch,'experiment':experiment,'source_repo':config['phase1_sources'][arch]['repo'],
              'source_weights':config['phase1_sources'][arch]['weight'], 'model_parameters':args,
              'tokenizer_size':len(tokenizer)}
    (work/'phase1_used.json').write_text(json.dumps(metadata,indent=2))
    backbone = instantiate(arch,args,student=student)
    load_pretrained(backbone,state,student=student)
    del state
    if arch=='a1':
        # As in the Phase 2 A1 notebook, classification uses hidden vectors,
        # not the costly full-vocabulary language-model output projection.
        backbone.output_layer = nn.Identity()
    model = RRClassifier(backbone,arch,dim,len(config['labels']))
    adapted = make_adaptation(model,experiment)
    ntrain = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(experiment,'loaded exactly:',str(weight),'| trainable:',f'{ntrain:,}','| adapters:',len(adapted))
    return model,tokenizer,weight

def tokenize_examples(examples,tok,labels,max_len,use_context):
    ids={name:i for i,name in enumerate(labels)}
    out=[]
    for ex in tqdm(examples,desc='Tokenize',leave=False):
        tokens,start,_=encode_example(ex,tok,max_len,use_context)
        out.append({'ids':tokens,'start':start,'gold':ids.get(ex['label'],-1),
                    'domain':ex['domain'],'doc_id':ex['doc_id'], 'sent_index':ex['sent_index']})
    return out


### 5.6 Batches, predictions and Macro-F1 metrics
We create mini-batches, calculate Macro-F1 and evaluate both Corporate Law and Information Technology documents.


In [ ]:
# PHASE3_SOURCE_CELL
def batches(examples,batch_size,shuffle=False):
    indices=list(range(len(examples)))
    if shuffle:random.shuffle(indices)
    for start in range(0,len(indices),batch_size):yield [examples[i] for i in indices[start:start+batch_size]]


def balanced_batches(examples, batch_size, shuffle=False):
    """Optionally sample rare labels more often during TRAINING only.

    Weight ~ count**(-0.25), limited to a 3x sampling advantage.
    The count uses ONLY training labels (never dev or test labels).
    Dev/test evaluation still uses every sentence exactly once, in order.
    """
    if not shuffle:
        yield from batches(examples, batch_size, shuffle=False)
        return
    counts = Counter(ex['gold'] for ex in examples)
    count_max = max(counts.values())
    sample_weights = []
    for ex in examples:
        ratio = (count_max / max(1, counts[ex['gold']])) ** 0.25
        sample_weights.append(min(3.0, ratio))
    weights = torch.tensor(sample_weights, dtype=torch.float)
    sampled = torch.multinomial(weights, num_samples=len(examples), replacement=True).tolist()
    for start in range(0, len(sampled), batch_size):
        yield [examples[i] for i in sampled[start:start + batch_size]]


def move_batch(batch,device):
    return ([torch.tensor(ex['ids'],device=device,dtype=torch.long) for ex in batch],
            [ex['start'] for ex in batch])


def scores(y_true,y_pred,nlabels):
    if not y_true:return {'macro_f1':0.0,'macro_precision':0.0,'macro_recall':0.0}
    p,r,f,_=precision_recall_fscore_support(y_true,y_pred,labels=list(range(nlabels)),
                                          average='macro',zero_division=0)
    return {'macro_f1':float(f),'macro_precision':float(p),'macro_recall':float(r)}


@torch.no_grad()
def predict_model(model,data,device,batch_size=4):
    model.eval();out=[]
    for batch in batches(data,batch_size,False):
        inputs,starts=move_batch(batch,device)
        out.extend(model(inputs,starts).float().argmax(-1).cpu().tolist())
    return out


def eval_model(model,dev,device,nlabels,batch_size):
    preds=predict_model(model,dev,device,batch_size)
    y=[x['gold'] for x in dev]
    overall=scores(y,preds,nlabels)
    domains={}
    for dom in ('CL','IT'):
        ii=[i for i,d in enumerate(dev) if d['domain']==dom]
        domains[dom]=scores([y[i] for i in ii],[preds[i] for i in ii],nlabels)
    return {'overall':overall,'domains':domains,'preds':preds}


### 5.7 Train, evaluate, and keep the best checkpoint

Training logs print Macro-F1 for CL, IT and both combined at the end of every epoch. The best development Macro-F1 weights are saved, even if training stops early. Rare-label balancing applies only to the training data, never test or development.

For A3, the student loss mixes ground-truth cross-entropy and a KL-divergence loss from the trained teacher.

In [ ]:
# PHASE3_SOURCE_CELL
def train_one(experiment,config,splits,labels,out_dir,device):
    arch=ARCH_BY_EXPERIMENT[experiment]
    work=Path(out_dir)/experiment;work.mkdir(parents=True,exist_ok=True)
    run_started=time.perf_counter()
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    model,tok,source_path=prepare_classifier(arch,experiment,config,student=experiment in ('a3_kd','a3_kd_improved'))
    max_len=int(config.get('max_length',160)); context=bool(config.get('use_context',True))
    train=tokenize_examples(make_examples(splits['train']),tok,labels,max_len,context)
    dev=tokenize_examples(make_examples(splits['dev']),tok,labels,max_len,context)
    test=tokenize_examples(make_examples(splits['test'],False),tok,labels,max_len,context)
    model.to(device)
    teacher=None
    if experiment in ('a3_kd','a3_kd_improved'):
        # Teacher is the *task-fine-tuned*, full-depth A3 LoRA teacher, not a random model.
        teacher_path=Path(out_dir)/'a3_teacher'/'best.pt'
        if not teacher_path.exists():
            raise FileNotFoundError('Run the A3 supervised teacher training before distillation.')
        teacher,tok_t,_=prepare_classifier('a3','a3_teacher',config)
        if len(tok_t)!=len(tok) or tok_t.get_vocab()!=tok.get_vocab():
            raise ValueError('Teacher and student tokenizer mapping mismatch')
        teacher.load_state_dict(torch.load(teacher_path, map_location='cpu', weights_only=True),strict=True)
        teacher.to(device).eval()
        for p in teacher.parameters():p.requires_grad=False
    # Phase 2 lesson: rare tags were ignored without some balancing. For Phase 3
    # use a modest inverse-sqrt class weighting (clipped for stability).
    # This uses ONLY train labels, never development or private test labels.
    label_counts=Counter(x['gold'] for x in train)
    total_labeled=len(train)
    class_weights=[]
    for label_id in range(len(labels)):
        n=label_counts.get(label_id,0)
        weight=math.sqrt(total_labeled / (len(labels)*max(n,1)))
        class_weights.append(max(0.5,min(3.0,weight)))
    class_weights=torch.tensor(class_weights,device=device,dtype=torch.float)
    # Student trains MORE parameters; use a smaller rate for stability.
    lr = (float(config.get('student_lr', 3e-5))
          if experiment in ('a3_kd', 'a3_kd_improved')
          else float(config.get('teacher_lr', config.get('lr', 2e-4))))
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                      lr=lr, weight_decay=0.01)
    print('Learning rate:', lr)
    print('Training samples:', len(train), '| Development samples:', len(dev))
    epochs=int(config.get('epochs',2)); batch_size=int(config.get('batch_size',4))
    grad_accum=int(config.get('grad_accum',4))
    patience=int(config.get('patience',2))
    best=-1;stale=0;history=[]
    for epoch in range(1,epochs+1):
        model.train();total_loss=0;steps=0
        optimizer.zero_grad(set_to_none=True)
        nbatches=math.ceil(len(train)/batch_size)
        # Rebalanced training batches are ONLY for training; development is untouched.
        train_iter = (balanced_batches(train,batch_size,True) if config.get('use_balanced_batches',True)
                      else batches(train,batch_size,True))
        for j,batch in enumerate(tqdm(train_iter,total=nbatches,desc=f'{experiment} epoch {epoch}')):
            inputs,starts=move_batch(batch,device)
            y=torch.tensor([x['gold'] for x in batch],device=device)
            with torch.autocast(device_type='cuda',dtype=torch.bfloat16,
                                enabled=device.type=='cuda' and torch.cuda.is_bf16_supported()):
                logits=model(inputs,starts)
                ce=F.cross_entropy(logits.float(),y,weight=class_weights)
                if teacher is None:loss=ce
                else:
                    with torch.no_grad():tlogits=teacher(inputs,starts)
                    temperature=2.0
                    kd=F.kl_div(F.log_softmax(logits.float()/temperature,dim=-1),
                                F.softmax(tlogits.float()/temperature,dim=-1),
                                reduction='batchmean')*temperature**2
                    loss=0.5*ce+0.5*kd
            (loss/grad_accum).backward();total_loss+=float(loss.detach());steps+=1
            if (j+1)%grad_accum==0 or j+1==nbatches:
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
                optimizer.step();optimizer.zero_grad(set_to_none=True)
        evaluation=eval_model(model,dev,device,len(labels),batch_size)
        metric=evaluation['overall']['macro_f1']
        print(f'{experiment} epoch {epoch}: train_loss={total_loss/max(1,steps):.4f} '
              f'dev Macro-F1={metric:.4f}, CL={evaluation["domains"]["CL"]["macro_f1"]:.4f} '
              f'IT={evaluation["domains"]["IT"]["macro_f1"]:.4f}')
        history.append({'epoch':epoch,'loss':total_loss/max(1,steps),
                        'dev':{k:v for k,v in evaluation.items() if k!='preds'}})
        if metric>best:
            best=metric;stale=0
            # Copy state tensors to CPU for saving; do not move live GPU model.
            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()}, work/'best.pt')
        else:
            stale+=1
            if stale>=patience:break
    model.load_state_dict(torch.load(work/'best.pt', map_location='cpu', weights_only=True),strict=True)
    model.to(device)
    ev=eval_model(model,dev,device,len(labels),batch_size)
    # Detailed validation metrics and confusion matrix for report.
    truth=[x['gold'] for x in dev];pred=ev.pop('preds')
    ev['per_label']=classification_report(truth,pred,labels=list(range(len(labels))),
                         target_names=labels,zero_division=0,output_dict=True)
    ev['confusion_matrix']=confusion_matrix(truth,pred,labels=list(range(len(labels)))).tolist()
    ev['history']=history
    # Controlled inference-time ablation: same trained model, but no previous sentence.
    noctx=tokenize_examples(make_examples(splits['dev']),tok,labels,max_len,False)
    ev['sentence_only_ablation']=eval_model(model,noctx,device,len(labels),batch_size)['overall']
    ev['elapsed_minutes']=round((time.perf_counter()-run_started)/60,2)
    ev['peak_cuda_gb']=(round(torch.cuda.max_memory_allocated()/1e9,3)
                      if torch.cuda.is_available() else None)
    ev['history']=history
    ev['training_source']=str(source_path)
    ev['parameter_count']=sum(p.numel() for p in model.parameters())
    ev['trainable_parameters']=sum(p.numel() for p in model.parameters() if p.requires_grad)
    ev['method']=experiment
    ev['best_macro_f1']=best
    ev['true_counts']={name:truth.count(i) for i,name in enumerate(labels)}
    ev['predicted_counts']={name:pred.count(i) for i,name in enumerate(labels)}
    ev['context']='previous + following sentence placed BEFORE target; causal last-token classification'
    print('True label counts:', ev['true_counts'])
    print('Predicted label counts:', ev['predicted_counts'])
    ev['class_weights']={name:float(class_weights[i]) for i,name in enumerate(labels)}
    (work/'metrics.json').write_text(json.dumps(ev,indent=2),encoding='utf8')
    with torch.no_grad():pred_test=predict_model(model,test,device,batch_size)
    submission=reassemble(splits['test'],test,pred_test,labels)
    validate_predictions(submission,splits['test'],labels)
    write_submission(submission,work)
    del model,teacher,optimizer
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return ev


def reassemble(test_docs,flat_test,preds,labels):
    assert len(flat_test)==len(preds)
    result={d['id']:[None]*len(d['text']) for d in test_docs}
    for ex,idx in zip(flat_test,preds):
        arr=result[ex['doc_id']]
        i=ex['sent_index']
        if arr[i] is not None:raise ValueError('Duplicate sentence prediction')
        arr[i]=labels[idx]
    if any(v is None for arr in result.values() for v in arr):raise ValueError('Missing sentence prediction')
    return result


def validate_predictions(predictions,test_docs,labels):
    ids={d['id'] for d in test_docs}
    if set(predictions)!=ids:raise ValueError(f'Wrong test ID set: missing {ids-set(predictions)}, extra {set(predictions)-ids}')
    allowed=set(labels)
    for d in test_docs:
        got=predictions[d['id']]
        if not isinstance(got,list) or len(got)!=len(d['text']):
            raise ValueError(f'Wrong sentence count for {d["id"]}')
        if any(not isinstance(x,str) or x not in allowed for x in got):
            raise ValueError(f'Invalid rhetorical role for {d["id"]}')
    return True


def write_submission(predictions,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    path=folder/'prediction.json'
    with open(path,'w',encoding='utf8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)
    with zipfile.ZipFile(folder/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.write(path,arcname='prediction.json')
    with zipfile.ZipFile(folder/'submission.zip') as z:
        assert z.namelist()==['prediction.json']
        assert json.loads(z.read('prediction.json'))==predictions


## 6. Download official Phase 3 data and verify labels
This uses the public CL/IT train/dev/test JSON files. The official `label_vocab.json` determines spelling; all 13 labels must match the published specification.


In [ ]:
# PHASE3_SOURCE_CELL
paths = download_data(OUTPUT_DIR / 'data', config['dataset_repo'])
splits, LABELS = load_splits(paths)
config['labels'] = LABELS

OFFICIAL_LABELS = {
    'Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute',
    'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon',
    'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision',
    'RulingByPresentCourt', 'None'
}
if set(LABELS) != OFFICIAL_LABELS:
    raise ValueError('Unexpected official role labels: ' + str(LABELS))
for name in ('train', 'dev', 'test'):
    print(name, 'documents:', len(splits[name]),
          '| sentences:', sum(len(d['text']) for d in splits[name]))
    assert splits[name], 'Empty Phase 3 split: ' + name
print('13 official labels:', LABELS)


## 7. Understand the data and check the pretrained checkpoint
Print one real legal sentence, inspect the role frequencies and verify that the tokenizer vocabulary size matches the saved model weights.


In [ ]:
# PHASE3_SOURCE_CELL
from collections import Counter
print('Sample text:', splits['train'][0]['text'][0][:300])
print('Its correct label:', splits['train'][0]['labels'][0])
counts = Counter(label for doc in splits['train'] for label in doc['labels'])
for name in LABELS:
    print(f'{name:28s} : {counts[name]}')

weight_file, tokenizer, saved_settings = load_phase1_files(ARCH, config)
state = _read_state(weight_file)
model_settings, hidden_size = actual_model_params(ARCH, state, saved_settings, len(tokenizer))
print('Using trained Phase 1 checkpoint:', weight_file)
print('Tokenizer vocab size:', len(tokenizer), '| hidden dimension:', hidden_size)
print('Architecture settings:', model_settings)
del state  # saves RAM before training


## 8. Train A4 LORMA

Start from my Phase 1 A4 checkpoint, train the small LoRMA+ adapters plus classification head, and save the best development checkpoint.

In [ ]:
# PHASE3_SOURCE_CELL
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable GPU before full Phase 3 training in Kaggle/Colab.')

# This runs fine-tuning, validation, final test predictions and ZIP creation.
metrics = train_one(EXPERIMENT, config, splits, LABELS, OUTPUT_DIR, DEVICE)
print('Best dev Macro-F1:', metrics['overall']['macro_f1'])


## 9. Verify metrics and Codabench output

The local Macro-F1 is computed on official public development labels, **not private test labels**. Print common errors and check that every official test document has one label for every sentence.

A higher score is NOT guaranteed; compare development results with your earlier runs.

In [ ]:
# PHASE3_SOURCE_CELL
import json, zipfile
exp_dir = OUTPUT_DIR / EXPERIMENT
metrics = json.loads((exp_dir / 'metrics.json').read_text(encoding='utf-8'))
print('Development Macro-F1:', metrics['overall']['macro_f1'])
print('Corporate Law Macro-F1:', metrics['domains']['CL']['macro_f1'])
print('Information Technology Macro-F1:', metrics['domains']['IT']['macro_f1'])
print('Sentence-only ablation F1:', metrics['sentence_only_ablation']['macro_f1'])
print('True labels on dev:', metrics.get('true_counts'))
print('Predicted labels on dev:', metrics.get('predicted_counts'))
print('Rare/difficult labels (lowest five dev F1):')
for lab in sorted(LABELS, key=lambda x: metrics['per_label'][x]['f1-score'])[:5]:
    print(' ', lab, round(metrics['per_label'][lab]['f1-score'], 4))

predictions = json.loads((exp_dir / 'prediction.json').read_text(encoding='utf-8'))
assert validate_predictions(predictions, splits['test'], LABELS)
with zipfile.ZipFile(exp_dir / 'submission.zip') as z:
    assert z.namelist() == ['prediction.json']
    assert json.loads(z.read('prediction.json')) == predictions
print('PASS: prediction.json contains exactly', len(predictions), 'official test IDs')
print('Codabench file:', exp_dir / 'submission.zip')


## 10. LAST CELL — upload my actual experiment results to Hugging Face

Run after training and all submission tests. The cell uploads the best checkpoint, metrics, prediction JSON, and Codabench ZIP to a **private** Hugging Face model repository. For A3 it uploads the teacher too. This uses the `HF` Kaggle secret; never paste a token in code.

In [ ]:
# Always run this LAST, after the actual training and ZIP verification.
# A private repo is used to avoid accidentally publishing trained files.
from pathlib import Path
from huggingface_hub import HfApi
import json

needed = ['best.pt', 'metrics.json', 'prediction.json', 'submission.zip', 'phase1_used.json']
for file_name in needed:
    if not (exp_dir / file_name).is_file():
        raise FileNotFoundError('Train and check results before uploading: ' + file_name)

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
if not api.repo_info(repo_id=OUTPUT_REPO, repo_type='model').private:
    raise RuntimeError('The target repository is not private. Upload cancelled for safety.')

# Save a source snapshot only from MY marked Phase 3 code cells.
# Never upload unrelated Kaggle history or authentication-cell content.
try:
    history = get_ipython().history_manager.input_hist_raw
    source_parts = []
    for raw_cell in history[1:]:
        if not raw_cell.lstrip().startswith('# PHASE3_SOURCE_CELL'):
            continue
        source_parts.append(raw_cell)
    source_path = exp_dir / 'phase3_source_snapshot.py'
    source_path.write_text('\n\n# ----- Next notebook cell -----\n\n'.join(source_parts), encoding='utf-8')
    print('Saved source snapshot:', source_path.name)
except Exception as error:
    print('Could not save a history snapshot:', error)

# Upload the entire experiment directory: best.pt, metrics, prediction.json,
# submission.zip, provenance and (where available) executable code snapshot.
api.upload_folder(folder_path=str(exp_dir), repo_id=OUTPUT_REPO,
                  repo_type='model', path_in_repo='results',
                  commit_message='Phase 3 experiment with results and source')

# Confirm that Hugging Face really received every required output.
remote_files = set(api.list_repo_files(repo_id=OUTPUT_REPO, repo_type='model'))
for file_name in needed:
    assert ('results/' + file_name) in remote_files, 'Not uploaded: ' + file_name
print('Verified private Hugging Face upload:', 'https://huggingface.co/' + OUTPUT_REPO)
print('Submit THIS zip separately to Codabench:', exp_dir / 'submission.zip')
